Today you will learn:   

* What happens inside a retriever
* Difference between similarity_search() and Retriever
* What is k
* Similarity scores
* Why retrieval sometimes fails
* MMR (Max Marginal Relevance)
* Compare Normal Retrieval vs MMR
  
No Gemini import is needed today because today we're focusing only on retrieval, not answer generation.

User Question    
      ↓   
Convert Question → Embedding   
      ↓    
Search Vector Database    
      ↓   
Retrieve Best Chunks   
      ↓   
    Gemini  
      ↓   
Final Answer

In [1]:
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

C:\Users\Dell\AppData\Local\Temp\ipykernel_1680\1814625698.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


**Load all PDFs**

In [2]:
folder = "day16_pdfs"

documents = []

for file in os.listdir (folder):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(folder, file))
        documents.extend(loader.load())

print("Pages Loade:", len(documents))

Pages Loade: 121


**check metadata**

In [3]:
print(documents[0].metadata)

{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'day16_pdfs\\attension-is-all-u-need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


**Split Documents**

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 863


**Create Embeddings**

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

**Create FAISS Vector Database**

In [6]:
vector_db = FAISS.from_documents(
    chunks,
    embedding=embeddings
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


**Create FAISS Vector Database**

In [7]:
vector_db = FAISS.from_documents(
    chunks,
    embedding=embeddings
)

print("Vector Database Created Successfully!")

Vector Database Created Successfully!


**Create Normal Retriever**

In [8]:
retriever = vector_db.as_retriever(
    search_kwargs={"k":3}
)

print("Normal Retriever Ready!")

Normal Retriever Ready!


**Similarity Search**

In [9]:
query = "What is BERT?"

docs = vector_db.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(docs):
    print("="*60)
    print(f"Document {i+1}")
    print(doc.page_content)

Document 1
asTi∈ RH.
For a given token, its input representation is
constructed by summing the corresponding token,
segment, and position embeddings. A visualiza-
tion of this construction can be seen in Figure 2.
3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditional left-to-right or
right-to-left language models to pre-train BERT.
Instead, we pre-train BERT using two unsuper-
vised tasks, described in this section. This step
Document 2
rameters as BERTBASE :
No NSP : A bidirectional model which is trained
using the “masked LM” (MLM) but without the
“next sentence prediction” (NSP) task.
LTR & No NSP: A left-context-only model which
is trained using a standard Left-to-Right (LTR)
LM, rather than an MLM. The left-only constraint
was also applied at ﬁne-tuning, because removing
it introduced a pre-train/ﬁne-tune mismatch that
degraded downstream performance. Additionally,
this model was pre-trained without the NSP task.
Document 3
GLUE le

**Understanding K**  

lets try different values of k

In [10]:
# k= 1
query = "What is BERT?"

docs = vector_db.similarity_search(
    query,
    k=1
)

for i, doc in enumerate(docs):
    print("="*60)
    print(f"Document {i+1}")
    print(doc.page_content)

Document 1
asTi∈ RH.
For a given token, its input representation is
constructed by summing the corresponding token,
segment, and position embeddings. A visualiza-
tion of this construction can be seen in Figure 2.
3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditional left-to-right or
right-to-left language models to pre-train BERT.
Instead, we pre-train BERT using two unsuper-
vised tasks, described in this section. This step


In [11]:
# k = 2
query = "What is BERT?"

docs = vector_db.similarity_search(
    query,
    k=2
)

for i, doc in enumerate(docs):
    print("="*60)
    print(f"Document {i+1}")
    print(doc.page_content)

Document 1
asTi∈ RH.
For a given token, its input representation is
constructed by summing the corresponding token,
segment, and position embeddings. A visualiza-
tion of this construction can be seen in Figure 2.
3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditional left-to-right or
right-to-left language models to pre-train BERT.
Instead, we pre-train BERT using two unsuper-
vised tasks, described in this section. This step
Document 2
rameters as BERTBASE :
No NSP : A bidirectional model which is trained
using the “masked LM” (MLM) but without the
“next sentence prediction” (NSP) task.
LTR & No NSP: A left-context-only model which
is trained using a standard Left-to-Right (LTR)
LM, rather than an MLM. The left-only constraint
was also applied at ﬁne-tuning, because removing
it introduced a pre-train/ﬁne-tune mismatch that
degraded downstream performance. Additionally,
this model was pre-trained without the NSP task.


In [24]:
# k = 4
query = "What is BERT?"

docs = vector_db.similarity_search(
    query,
    k=4
)

for i, doc in enumerate(docs):
    print("="*60)
    print(f"Document {i+1}")
    print(doc.page_content)

Document 1
asTi∈ RH.
For a given token, its input representation is
constructed by summing the corresponding token,
segment, and position embeddings. A visualiza-
tion of this construction can be seen in Figure 2.
3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditional left-to-right or
right-to-left language models to pre-train BERT.
Instead, we pre-train BERT using two unsuper-
vised tasks, described in this section. This step
Document 2
rameters as BERTBASE :
No NSP : A bidirectional model which is trained
using the “masked LM” (MLM) but without the
“next sentence prediction” (NSP) task.
LTR & No NSP: A left-context-only model which
is trained using a standard Left-to-Right (LTR)
LM, rather than an MLM. The left-only constraint
was also applied at ﬁne-tuning, because removing
it introduced a pre-train/ﬁne-tune mismatch that
degraded downstream performance. Additionally,
this model was pre-trained without the NSP task.
Document 3
GLUE le

In [25]:
# k = 5
query = "What is BERT?"

docs = vector_db.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(docs):
    print("="*60)
    print(f"Document {i+1}")
    print(doc.page_content)

Document 1
asTi∈ RH.
For a given token, its input representation is
constructed by summing the corresponding token,
segment, and position embeddings. A visualiza-
tion of this construction can be seen in Figure 2.
3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use traditional left-to-right or
right-to-left language models to pre-train BERT.
Instead, we pre-train BERT using two unsuper-
vised tasks, described in this section. This step
Document 2
rameters as BERTBASE :
No NSP : A bidirectional model which is trained
using the “masked LM” (MLM) but without the
“next sentence prediction” (NSP) task.
LTR & No NSP: A left-context-only model which
is trained using a standard Left-to-Right (LTR)
LM, rather than an MLM. The left-only constraint
was also applied at ﬁne-tuning, because removing
it introduced a pre-train/ﬁne-tune mismatch that
degraded downstream performance. Additionally,
this model was pre-trained without the NSP task.
Document 3
GLUE le

**Similarity Search with Scores**

In [14]:
results = vector_db.similarity_search_with_score(
    "What is BERT?",
    k=5
)

for doc, score in results:
    print("="*60)
    print("Score:", score)
    print(doc.metadata["source"])
    print(doc.page_content[:300])

Score: 0.9663217
day16_pdfs\bert.pdf
asTi∈ RH.
For a given token, its input representation is
constructed by summing the corresponding token,
segment, and position embeddings. A visualiza-
tion of this construction can be seen in Figure 2.
3.1 Pre-training BERT
Unlike Peters et al. (2018a) and Radford et al.
(2018), we do not use tradi
Score: 0.9706633
day16_pdfs\bert.pdf
rameters as BERTBASE :
No NSP : A bidirectional model which is trained
using the “masked LM” (MLM) but without the
“next sentence prediction” (NSP) task.
LTR & No NSP: A left-context-only model which
is trained using a standard Left-to-Right (LTR)
LM, rather than an MLM. The left-only constraint
was
Score: 0.9729637
day16_pdfs\bert.pdf
GLUE leaderboard10, BERTLARGE obtains a score
of 80.5, compared to OpenAI GPT, which obtains
72.8 as of the date of writing.
We ﬁnd that BERT LARGE signiﬁcantly outper-
forms BERTBASE across all tasks, especially those
with very little training data. The effect of model
size is explored

**Lower score generally means a closer match (depending on the vector store implementation).**

**try differnent question**

In [15]:
results = vector_db.similarity_search_with_score(
    "Who developed Gemini?",
    k=5
)

for doc, score in results:
    print("="*60)
    print("Score:", score)
    print(doc.metadata["source"])
    print(doc.page_content[:300])

Score: 0.63992786
day16_pdfs\gemini.pdf
Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like
Score: 0.7118453
day16_pdfs\gemini.pdf
Gemini: A Family of Highly Capable Multimodal Models
Terzi, Vladimir Mikulik, Igor Babuschkin, Aidan Clark, Diego de Las Casas, Aurelia Guy, Chris Jones,
James Bradbury, Matthew Johnson, Blake A. Hechtman, Laura Weidinger, Iason Gabriel, William S.
Isaac, Edward Lockhart, Simon Osindero, Laura Rimel
Score: 0.78886235
day16_pdfs\gemini.pdf
Gemini: A Family of Highly Capable Multimodal Models
Daniel Salz, Mario Lucic, Michael Tschannen, Arsha Nagrani, Hexiang Hu, Mandar Joshi, Bo Pang,
Ceslee Montgomery, Paulina Pietrzyk, Marvin Ritter, AJ Piergiovanni, Matthias Minderer, Filip
Pavetic, Austin Waters, Gang Li, Ibra

**MMR Retrieval**   
Instead of returning very similar chunks from the same page, MMR tries to return diverse but relevant chunks.

In [17]:
mmr_retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":3,
        "fetch_k":10
    }
)

**Test MMR**

In [18]:
query = "Who developed Gemini?"

docs = mmr_retriever.invoke(query)

**Display MMR results**

In [19]:
for i, doc in enumerate(docs):

    print("="*70)

    print(f"Document {i+1}")

    print(doc.metadata["source"])

    print()

    print(doc.page_content)

Document 1
day16_pdfs\gemini.pdf

Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like to acknowledge the support from Abhi Mohan, Adekunle Bello, Aishwarya Nagarajan, Alaa
Saade, Alejandro Lince, Alexander Chen, Alexander Kolbasov, Alexander Schiffhauer, Ameya Shringi,
Document 2
day16_pdfs\gemini.pdf

each role does not indicate ordering of the contributions.
Gemini is a cross-Google effort, with members from Google DeepMind (GDM), Google Research
(GR), Bard/Assistant, Knowledge and Information (K&I), Core ML, Cloud, Labs, and more.
We thank Aakanksha Chowdhery, Dustin Tran, Heng-Tze Cheng, Jack W. Rae, Kate Olszewska,
Mariko Iinuma, Peter Humphreys, Shashi Narayan, and Steven Zheng for leading the preparation of
Document 3
day16_pdfs\gemin

**Compare Both Retrievers**

In [20]:
#normal retriever
normal_docs = retriever.invoke(
    "Who developed Gemini?"
)

In [21]:
#MMR retriever
mmr_docs = mmr_retriever.invoke(
    "Who developed Gemini?"
)

**Display normal results**

In [22]:
print("NORMAL RETRIEVER\n")

for doc in normal_docs:

    print("="*60)

    print(doc.metadata["source"])

    print(doc.page_content)

NORMAL RETRIEVER

day16_pdfs\gemini.pdf
Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like to acknowledge the support from Abhi Mohan, Adekunle Bello, Aishwarya Nagarajan, Alaa
Saade, Alejandro Lince, Alexander Chen, Alexander Kolbasov, Alexander Schiffhauer, Ameya Shringi,
day16_pdfs\gemini.pdf
each role does not indicate ordering of the contributions.
Gemini is a cross-Google effort, with members from Google DeepMind (GDM), Google Research
(GR), Bard/Assistant, Knowledge and Information (K&I), Core ML, Cloud, Labs, and more.
We thank Aakanksha Chowdhery, Dustin Tran, Heng-Tze Cheng, Jack W. Rae, Kate Olszewska,
Mariko Iinuma, Peter Humphreys, Shashi Narayan, and Steven Zheng for leading the preparation of
day16_pdfs\gemini.pdf
Gemini: A F

**Display MMR results**

In [23]:
print("MMR RETRIEVER\n")

for doc in mmr_docs:

    print("="*60)

    print(doc.metadata["source"])

    print(doc.page_content)

MMR RETRIEVER

day16_pdfs\gemini.pdf
Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like to acknowledge the support from Abhi Mohan, Adekunle Bello, Aishwarya Nagarajan, Alaa
Saade, Alejandro Lince, Alexander Chen, Alexander Kolbasov, Alexander Schiffhauer, Ameya Shringi,
day16_pdfs\gemini.pdf
each role does not indicate ordering of the contributions.
Gemini is a cross-Google effort, with members from Google DeepMind (GDM), Google Research
(GR), Bard/Assistant, Knowledge and Information (K&I), Core ML, Cloud, Labs, and more.
We thank Aakanksha Chowdhery, Dustin Tran, Heng-Tze Cheng, Jack W. Rae, Kate Olszewska,
Mariko Iinuma, Peter Humphreys, Shashi Narayan, and Steven Zheng for leading the preparation of
day16_pdfs\gemini.pdf
Gemini: A Fami

In [26]:
query = "Who developed Gemini?"

results = vector_db.similarity_search_with_score(query, k=10)

for i, (doc, score) in enumerate(results):
    print("="*70)
    print(f"Rank: {i+1}")
    print(f"Score: {score:.4f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Page: {doc.metadata['page']}")
    print(doc.page_content[:300])

Rank: 1
Score: 0.6399
Source: day16_pdfs\gemini.pdf
Page: 69
Gemini: A Family of Highly Capable Multimodal Models
Beroshi, Joel Moss, Jon Small, Jonathan Fildes, Kathy Meier-Hellstern, Lisa Patel, Oli Gaymond,
Rebecca Bland, Reena Jana, Tessa Lueth, and Tom Lue.
Our work is made possible by the dedication and efforts of numerous teams at Google. We would
like
Rank: 2
Score: 0.7118
Source: day16_pdfs\gemini.pdf
Page: 48
Gemini: A Family of Highly Capable Multimodal Models
Terzi, Vladimir Mikulik, Igor Babuschkin, Aidan Clark, Diego de Las Casas, Aurelia Guy, Chris Jones,
James Bradbury, Matthew Johnson, Blake A. Hechtman, Laura Weidinger, Iason Gabriel, William S.
Isaac, Edward Lockhart, Simon Osindero, Laura Rimel
Rank: 3
Score: 0.7889
Source: day16_pdfs\gemini.pdf
Page: 42
Gemini: A Family of Highly Capable Multimodal Models
Daniel Salz, Mario Lucic, Michael Tschannen, Arsha Nagrani, Hexiang Hu, Mandar Joshi, Bo Pang,
Ceslee Montgomery, Paulina Pietrzyk, Marvin Ritter, AJ Piergiovanni